In [1]:
import os
import numpy as np
import torch
import torch.nn as nn

In [2]:
class LinearProjection(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(LinearProjection, self).__init__()
        self.linear = nn.Linear(input_dim, output_dim)

    def forward(self, x):
        return self.linear(x)

In [3]:
def load_embedding(filepath):
    """Charge un embedding à partir d'un fichier .txt."""
    try:
        return np.loadtxt(filepath)
    except Exception as e:
        print(f"Erreur lors du chargement de l'embedding '{filepath}' : {e}")
        return None

def save_embedding(filepath, embedding):
    """Sauvegarde un embedding dans un fichier .txt."""
    try:
        np.savetxt(filepath, embedding)
    except Exception as e:
        print(f"Erreur lors de la sauvegarde de l'embedding '{filepath}' : {e}")

In [4]:
def generate_multimodal_embeddings(vgg16_dir='VGG16', text_extracted_dir='TEXT_EXTRACTED', multimodal_output_dir='MULTIMODAL'):
    # 0. Vérifier l'existence des dossiers d'entrée
    if not os.path.exists(vgg16_dir):
        print(f"Erreur : Le dossier '{vgg16_dir}' est introuvable. Veuillez le créer et y placer les embeddings VGG16.")
        return
    if not os.path.exists(text_extracted_dir):
        print(f"Erreur : Le dossier '{text_extracted_dir}' est introuvable. Veuillez le créer et y placer les embeddings textuels.")
        return

    # 1. Créer le dossier de sortie pour les embeddings multimodaux
    os.makedirs(multimodal_output_dir, exist_ok=True)
    print(f"Le dossier de sortie '{multimodal_output_dir}' a été créé ou existe déjà.")

    # Définir les dimensions attendues
    VGG16_DIM = 768  # Dimension attendue pour les embeddings VGG16
    TEXT_DIM = 384   # Dimension attendue pour les embeddings textuels (Sentence-BERT par défaut)
    CONCATENATED_DIM = TEXT_DIM + TEXT_DIM # Dimension après concaténation (384 + 384 = 768)

    # Initialiser le modèle de projection linéaire
    # Nous projetons VGG16 de 768 à 384 dimensions
    projection_model = LinearProjection(VGG16_DIM, TEXT_DIM)
    # Il est souvent utile de charger un modèle pré-entraîné ici si vous en avez un,
    # sinon, il sera entraîné avec des poids aléatoires et devra être optimisé.
    # Pour cette tâche, nous allons simplement utiliser les poids aléatoires car il n'y a pas d'entraînement ici.
    print(f"Modèle de projection linéaire créé : {VGG16_DIM} -> {TEXT_DIM} dimensions.")

    # Obtenir la liste des fichiers d'embeddings VGG16
    vgg16_files = {os.path.splitext(f)[0].replace('_visual', ''): os.path.join(vgg16_dir, f)
                   for f in os.listdir(vgg16_dir) if f.endswith('.txt')}

    # Initialiser les compteurs
    processed_count = 0
    missing_text_files = 0
    missing_vgg16_files = 0
    skipped_files = 0

    print("\nDébut de la génération des embeddings multimodaux...")

    # Parcourir les fichiers d'embeddings textuels (car ils sont souvent la référence)
    for text_filename in os.listdir(text_extracted_dir):
        if text_filename.endswith('.txt'):
            # Obtenir le nom de base de l'image (sans extension .txt)
            base_image_name = os.path.splitext(text_filename)[0]

            text_filepath = os.path.join(text_extracted_dir, text_filename)
            vgg16_filepath = vgg16_files.get(base_image_name) # Chercher le fichier VGG16 correspondant

            if vgg16_filepath is None:
                print(f"Avertissement : Fichier VGG16 manquant pour l'image '{base_image_name}'. Le fichier textuel '{text_filename}' sera ignoré.")
                missing_vgg16_files += 1
                skipped_files += 1
                continue

            # Charger les embeddings
            text_embedding = load_embedding(text_filepath)
            vgg16_embedding = load_embedding(vgg16_filepath)

            if text_embedding is None or vgg16_embedding is None:
                skipped_files += 1
                continue # Passer à l'image suivante si un chargement a échoué

            # Vérifier les dimensions des embeddings chargés
            if text_embedding.shape[0] != TEXT_DIM:
                print(f"Avertissement : Dimension inattendue pour l'embedding texte '{text_filename}' : {text_embedding.shape[0]}. Attendu : {TEXT_DIM}. Fichier ignoré.")
                skipped_files += 1
                continue
            if vgg16_embedding.shape[0] != VGG16_DIM:
                print(f"Avertissement : Dimension inattendue pour l'embedding VGG16 '{os.path.basename(vgg16_filepath)}' : {vgg16_embedding.shape[0]}. Attendu : {VGG16_DIM}. Fichier ignoré.")
                skipped_files += 1
                continue

            # Convertir l'embedding VGG16 en tenseur PyTorch et le projeter
            vgg16_embedding_tensor = torch.from_numpy(vgg16_embedding).float()
            projected_vgg16_embedding_tensor = projection_model(vgg16_embedding_tensor)
            projected_vgg16_embedding = projected_vgg16_embedding_tensor.detach().numpy() # Reconvertir en numpy
            multimodal_embedding = np.concatenate((projected_vgg16_embedding, text_embedding))

            # Vérifier la dimension finale
            if multimodal_embedding.shape[0] != CONCATENATED_DIM:
                print(f"Erreur interne : Dimension finale inattendue pour '{base_image_name}'. Attendu : {CONCATENATED_DIM}, Obtenu : {multimodal_embedding.shape[0]}.")
                skipped_files += 1
                continue

            # Sauvegarder l'embedding multimodal
            output_multimodal_filepath = os.path.join(multimodal_output_dir, f"{base_image_name}.txt")
            save_embedding(output_multimodal_filepath, multimodal_embedding)
            processed_count += 1

            if processed_count % 100 == 0:
                print(f"  {processed_count} embeddings multimodaux traités...")

    print(f"\n----------------------------------------------------------------------")
    print(f"Processus terminé.")
    print(f"Total des embeddings multimodaux générés : {processed_count}")
    if missing_text_files > 0:
        print(f"Nombre de fichiers textuels manquants dans VGG16 : {missing_text_files}")
    if missing_vgg16_files > 0:
        print(f"Nombre de fichiers VGG16 manquants pour des fichiers textuels : {missing_vgg16_files}")
    if skipped_files > 0:
        print(f"Nombre de fichiers ignorés en raison d'erreurs ou de dimensions incorrectes : {skipped_files}")
    print(f"Vérifiez le dossier '{multimodal_output_dir}' pour les fichiers de sortie.")


In [5]:
if __name__ == "__main__":
    generate_multimodal_embeddings()

Le dossier de sortie 'MULTIMODAL' a été créé ou existe déjà.
Modèle de projection linéaire créé : 768 -> 384 dimensions.

Début de la génération des embeddings multimodaux...
  100 embeddings multimodaux traités...
  200 embeddings multimodaux traités...
  300 embeddings multimodaux traités...
  400 embeddings multimodaux traités...
  500 embeddings multimodaux traités...
  600 embeddings multimodaux traités...
  700 embeddings multimodaux traités...
  800 embeddings multimodaux traités...
  900 embeddings multimodaux traités...
  1000 embeddings multimodaux traités...
  1100 embeddings multimodaux traités...
  1200 embeddings multimodaux traités...
  1300 embeddings multimodaux traités...
  1400 embeddings multimodaux traités...
  1500 embeddings multimodaux traités...
  1600 embeddings multimodaux traités...
  1700 embeddings multimodaux traités...
  1800 embeddings multimodaux traités...
  1900 embeddings multimodaux traités...
  2000 embeddings multimodaux traités...
  2100 embeddi